Build a Simple LLM Application with LCEL

In this quickstart we'll show you how to build a simple LLM application with LangChain. This application will translate text from English into another language. This is a relatively simple LLM application - it's just just a single LLM call plus some prompting. Still, this is a great way to get started with LangChain - a lot of features can be built with just some prompting and an LLm call!

After this you will have a high level overview of :

- Using language models
- Using PromptTemplates and OutputParsers
- Using LangChain Expression Language (LCEL) to chain components together
- Debugging and tracing your application using LangSmith
- Deploying your application with LangServe

In [7]:
!pip install langchain

In [8]:
### For this project we will be using Open Source models -- Llama3, Gemma2, Mistral
# Groq --> a hardware accelerator / chip optimized for running LLMs. Can significantly speed up inference, especially for large models like Llama3 or Mistral.

###### Groq is a high-performance LPU (Language Processing Unit)–based inference engine designed specifically for running large language models at extremely high speed and low latency.
###### Unlike GPUs that are built for general-purpose parallel computation, Groq’s LPU is optimized entirely for deterministic, sequential matrix operations required during LLM inference.

###### This allows Groq to achieve:

###### - Very high tokens-per-second generation

###### - Consistent, predictable latency

###### - Efficient execution of open-source LLMs like Llama, Mistral, and Gemma

In [9]:
## What is the LPU Inference Engine ?

## An LPU Inference Engine, with LPU standing for Language Processing Unit, is a hardware and software platform that delivers exceptional compute speed, quality, and energy efficiency. This new type of end-to-end processing unit system provides the fastest inference for computationally intensive applications with sequential components, such as AI language applications like Large Language Models(LLMs).

## Why is it so much faster than GPUs for LLMs and GenAI?

## The LPU is designed to overcome the two LLM bottlenecks : compute density and memory bandwidth. An LPU has greater compute capacity than a GPU and CPU in regards to LLMs. This reduces the amount of time per word calculated, allowing sequences of text to be generated much faster. Additionally, eliminating external memory bottlenecks enables the LPU Inference Engine to deliver orders of magnitude better performance on LLMs compared to GPUs.

In [10]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

groq_api_key

'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'

In [11]:
!pip install langchain_groq

In [20]:
from langchain_groq import ChatGroq

model = ChatGroq(model = "llama-3.1-8b-instant", groq_api_key = groq_api_key)

model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x00000281929B2E40>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000281929B3980>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [21]:
!pip install langchain_core

In [23]:
from langchain_core.messages import HumanMessage, SystemMessage

messages = [
    SystemMessage(content = "Translate the following from English to French"),
    HumanMessage(content = "Hello, how are you?")
]

result = model.invoke(messages)

In [24]:
result

AIMessage(content='Bonjour, comment allez-vous ?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 48, 'total_tokens': 56, 'completion_time': 0.021297639, 'completion_tokens_details': None, 'prompt_time': 0.002284298, 'prompt_tokens_details': None, 'queue_time': 0.053692411, 'total_time': 0.023581937}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_ff2b098aaf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019af3e7-6fd0-7932-808d-ebb7449dbd57-0', usage_metadata={'input_tokens': 48, 'output_tokens': 8, 'total_tokens': 56})

In [26]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()
parser.invoke(result)

'Bonjour, comment allez-vous ?'

In [27]:
## Using LCEL - we can chain the components

chain = model | parser
chain.invoke(messages)

'Bonjour, comment allez-vous ?'

In [28]:
## Prompt Templates

from langchain_core.prompts import ChatPromptTemplate

generic_template = "Translate the following into {language}:"

prompt = ChatPromptTemplate.from_messages(
    [("system", generic_template),("user","{text}")]
)

In [37]:
response = prompt.invoke({"language":"French","text":"Hello"})

In [39]:
response.to_messages()

[SystemMessage(content='Translate the following into French:', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello', additional_kwargs={}, response_metadata={})]

In [41]:
## Chaining together components with LCEL

chain1 = prompt | model | parser
chain1.invoke({"language":"French","text":"Hello"})

'Bonjour'

###### Chaining in LangChain means connecting multiple LLM steps—like the prompt, the model, and the output parser—into a single pipeline using the `|` operator. Instead of manually formatting prompts, calling the model, and parsing the response every time, a chain bundles these steps into one reusable function that can be invoked easily. This makes the code cleaner, avoids repetition, and allows LangServe or FastAPI to expose the entire workflow as an API endpoint. Essentially, chaining turns several LLM operations into one smooth, modular, and efficient process.
